# SHIELD — Master Notebook (Corrected Validation + Editable SMOTE Strength)

**PART A — Preprocessing:** clean data → target labelling → split → Information Gain (top 20) → SMOTE.

**PART B — Development:** Random Forest and XGBoost across all eight protocols: 70:30 only, 80:20 only,
CV=5 only, CV=10 only, 70:30+CV=5, 70:30+CV=10, 80:20+CV=5, 80:20+CV=10. **16 experiments.**

**Approved changes:** IG and SMOTE are fitted inside each CV training fold. CV-only experiments use nested CV.
One global `SMOTE_RATIO` starts at **0.1**; change it later to **0.25**, **0.5**, or **1.0** and rerun from Setup.

**On hold:** threshold tuning and additional RF hyperparameters. All runs keep the default binary decision rule
from `predict()` (approximately a 0.5 cutoff). No `min_samples_leaf` or `max_features` search is added.

**Output style preserved:** Accuracy, Precision, Recall, F1-Score, ROC-AUC; the same one-line summary,
classification reports, grouped metric bars, paired confusion matrices, ROC curves, and CSV result tables.
Metrics for precision, recall, and F1 refer to the **Bankrupt (1)** class.

**Run instructions:** upload this notebook to Colab, verify `RAW_PATH`, then use **Runtime → Run all**.
For another SMOTE ratio, change the variable in A1 and rerun all cells. Outputs and checkpoints are isolated
by ratio, data fingerprint, configuration, and package versions. The original notebook/results are preserved.
Nested CV performs many fits and may take substantial time; completed outer folds are checkpointed.


---
# PART A — PREPROCESSING
Part A demonstrates one 80:20 split. Part B fits fresh pipelines directly on the original cleaned features;
it does **not** use the demo's feature-selected or SMOTE-resampled CSVs as CV inputs.


## A1. Setup and the editable SMOTE variable


In [ ]:
!pip install -q imbalanced-learn xgboost openpyxl

import os
import random
import json
import hashlib
import platform
import tempfile
import importlib.metadata
from functools import partial
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            roc_auc_score, roc_curve, confusion_matrix, classification_report)
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

# ==========================================================
# CHANGE THIS ONE VARIABLE TO RUN A DIFFERENT SMOTE STRENGTH.
# 0.1 = minority count / majority count after resampling = 1:10.
# Later comparisons: 0.25, 0.5, 1.0. This is NOT a probability threshold.
SMOTE_RATIO = 0.1
# ==========================================================

SEED = 42
K_FEATURES = 20
SMOTE_K_NEIGHBORS = 5
N_JOBS = -1  # GridSearchCV parallelism; each individual model uses one worker.
USE_PIPELINE_CACHE = True  # Reuse IG/SMOTE fits for identical training folds across candidates.
NOTEBOOK_VERSION = "shield-validated-smote-v1"
TARGET_COL = "Bankrupt"
LABEL_MAP = {0: "Not Bankrupt", 1: "Bankrupt"}

assert 0 < SMOTE_RATIO <= 1, "SMOTE_RATIO must be greater than 0 and at most 1."
random.seed(SEED)
np.random.seed(SEED)

drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/SHIELD_Project/data'
BASE_REPORT_DIR = '/content/drive/MyDrive/SHIELD_Project/reports'
BASE_MODEL_DIR = '/content/drive/MyDrive/SHIELD_Project/models'
RAW_PATH = f'{drive_path}/bankruptcy_italian_companies_2023.xlsx'
PACKAGE_VERSIONS = {
    name: importlib.metadata.version(name)
    for name in ['numpy', 'pandas', 'scikit-learn', 'imbalanced-learn', 'xgboost', 'joblib']
}
print('Setup complete. SEED =', SEED)
print('SMOTE_RATIO =', SMOTE_RATIO, '(minority / majority after resampling)')
print('Package versions:', PACKAGE_VERSIONS)


## A2. Load Raw Data
*Verification: shape and first rows of the untouched raw dataset.*


In [ ]:
df_raw = pd.read_excel(RAW_PATH)
print('=== RAW DATA LOADED ===')
print('Shape:', df_raw.shape)
display(df_raw.head())


## A3. Clean Data
Drop `Company` and constant predictor columns, matching the original cleaning flow. Validate the target,
numeric predictors, missing values, and infinities. Report duplicates without silently deleting companies
that happen to share financial ratios. No additional outlier removal or scaling is introduced in this version.


In [ ]:
df = df_raw.copy()
before_shape = df.shape
assert TARGET_COL in df.columns, f'Missing target column: {TARGET_COL}'
dropped_cols = []
if 'Company' in df.columns:
    df = df.drop(columns=['Company'])
    dropped_cols.append('Company')
constant_cols = [c for c in df.columns if c != TARGET_COL and df[c].nunique(dropna=False) <= 1]
if constant_cols:
    df = df.drop(columns=constant_cols)
    dropped_cols.extend(constant_cols)

assert df.isnull().sum().sum() == 0, 'Missing values found. Investigate before continuing.'
assert set(df[TARGET_COL].unique()) == {0, 1}, 'Target must contain exactly 0 and 1.'
predictor_cols = df.columns.drop(TARGET_COL)
non_numeric = df[predictor_cols].select_dtypes(exclude=[np.number]).columns.tolist()
assert not non_numeric, f'Non-numeric predictors found: {non_numeric}'
assert np.isfinite(df[predictor_cols].to_numpy(dtype=float)).all(), 'Infinite predictor values found.'
assert len(predictor_cols) >= K_FEATURES, 'Fewer predictors than K_FEATURES.'

print('=== DATA CLEANING VERIFICATION ===')
print(f'Shape before cleaning: {before_shape}')
print(f'Shape after cleaning:  {df.shape}')
print(f'Columns dropped ({len(dropped_cols)}): {dropped_cols}')
print('No missing values confirmed.')
duplicate_count = int(df.duplicated().sum())
print(f'Duplicate cleaned rows: {duplicate_count} (reported; not automatically removed)')
if 'Company' in df_raw:
    repeated_ids = int(df_raw['Company'].duplicated().sum())
    print(f'Repeated Company identifiers: {repeated_ids}')
    if repeated_ids:
        raise ValueError('Repeated Company identifiers require investigation/grouped splitting before continuing.')


## A4. Target Column and Labelling
`Bankrupt`: 0 = Not Bankrupt; 1 = Bankrupt.


In [ ]:
y_full = df[TARGET_COL].astype(int)
X_full = df.drop(columns=[TARGET_COL])
label_counts = y_full.value_counts().sort_index()
assert label_counts[1] < label_counts[0], 'This notebook expects Bankrupt (1) to be the minority class.'
assert SMOTE_RATIO > label_counts[1] / label_counts[0], 'SMOTE_RATIO is below the existing minority/majority ratio.'

print('=== TARGET LABELLING VERIFICATION ===')
for cls, count in label_counts.items():
    print(f'  {cls} ({LABEL_MAP[cls]}): {count} rows ({count / len(y_full) * 100:.2f}%)')
print(f'Total rows: {len(y_full)}')
print(f'Features (X) shape: {X_full.shape}')
print(f'Imbalance ratio (Bankrupt:Not Bankrupt) = 1:{label_counts[0] / label_counts[1]:.1f}')

# Hash the actual cleaned data, including feature names, order, dtypes, and values.
data_hasher = hashlib.sha256()
data_hasher.update(pd.util.hash_pandas_object(df, index=True).values.tobytes())
data_hasher.update(json.dumps([(str(c), str(df[c].dtype)) for c in df.columns]).encode())
DATA_FINGERPRINT = data_hasher.hexdigest()
print('Cleaned data fingerprint:', DATA_FINGERPRINT[:16])


## A5. Train-Test Split (80:20 demonstration)
*Verification: shapes and stratified class distributions.*


In [ ]:
X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X_full, y_full, test_size=0.20, random_state=SEED, stratify=y_full)
print('=== SPLIT VERIFICATION (80:20 demo) ===')
print(f'X_train: {X_train_demo.shape}, X_test: {X_test_demo.shape}')
print('y_train distribution:'); print(y_train_demo.value_counts().sort_index())
print('y_test distribution:'); print(y_test_demo.value_counts().sort_index())


## A6. Information Gain — Top 20 Features (demonstration)
Fit only on the demo training set, then transform the test set. The development pipeline repeats this
selection within each CV training fold. The final selected 20 may differ between folds and experiments;
the deployed final model has one saved, fixed feature list.


In [ ]:
def make_selector():
    return SelectKBest(score_func=partial(mutual_info_classif, random_state=SEED), k=K_FEATURES).set_output(transform='pandas')

selector_demo = make_selector()
X_train_sel_demo = selector_demo.fit_transform(X_train_demo, y_train_demo)
X_test_sel_demo = selector_demo.transform(X_test_demo)
selected_cols_demo = X_full.columns[selector_demo.get_support()].tolist()
ig_full_df = pd.DataFrame({'Feature': X_full.columns, 'IG_Score': selector_demo.scores_})
ig_full_df = ig_full_df.sort_values(by='IG_Score', ascending=False).reset_index(drop=True)

print('=== INFORMATION GAIN VERIFICATION (80:20 demo) ===')
print(f'Selected {len(selected_cols_demo)} features (target: {K_FEATURES})')
print('\nTop 20 Selected Features (ranked by IG score):')
display(ig_full_df.head(K_FEATURES))
print(f'\nX_train_selected shape: {X_train_sel_demo.shape}')
print(f'X_test_selected shape:  {X_test_sel_demo.shape}')


## A7. SMOTE — Training Set Only (demonstration)
Use the global ratio, not automatic 50:50 balancing. At `0.1`, the resampled minority count is about
one tenth of the majority count. The test set remains untouched.


In [ ]:
print('=== SMOTE VERIFICATION (80:20 demo) ===')
print('SMOTE_RATIO:', SMOTE_RATIO)
print('Train set BEFORE SMOTE:')
print(y_train_demo.value_counts().sort_index())
X_train_bal_demo, y_train_bal_demo = SMOTE(
    sampling_strategy=SMOTE_RATIO, random_state=SEED, k_neighbors=SMOTE_K_NEIGHBORS
).fit_resample(X_train_sel_demo, y_train_demo)
print('\nTrain set AFTER SMOTE:')
print(pd.Series(y_train_bal_demo).value_counts().sort_index())
print(f'\nTrain shape before: {X_train_sel_demo.shape} -> after: {X_train_bal_demo.shape}')
print('\nTest set (UNCHANGED, still imbalanced):')
print(y_test_demo.value_counts().sort_index())
assert len(X_test_sel_demo) == len(X_test_demo)


## A8. Preprocessing Summary


In [ ]:
print('=== PREPROCESSING PIPELINE SUMMARY ===')
print(f'1. Raw data loaded:        {df_raw.shape}')
print(f'2. After cleaning:         {df.shape} (dropped: {dropped_cols})')
print(f'3. Target labelled:        {TARGET_COL} -> 0/1 ({LABEL_MAP})')
print(f'4. Split (80:20 demo):     train {X_train_demo.shape}, test {X_test_demo.shape}')
print(f'5. IG top-{K_FEATURES}:            selected {len(selected_cols_demo)} features')
print(f'6. SMOTE (train only):     {X_train_sel_demo.shape} -> {X_train_bal_demo.shape}')
print('\nDevelopment uses X_full/y_full, not these preprocessed demo arrays.')


---
# PART B — DEVELOPMENT

**Split only:** train one pipeline on the training partition with the original fixed model parameters;
evaluate on the untouched test partition.

**Split + CV:** reserve the test partition; GridSearchCV fits IG → SMOTE → model separately within each
training fold, chooses parameters using Bankrupt-class F1, then refits on the entire training partition.
Report final metrics on the test partition. `CV_F1_Mean (%)` is the parameter-selection score.

**CV only:** use **nested stratified CV**. Each outer held-out fold is evaluated using a model selected by
an inner GridSearchCV on the outer training data only. CV=5 uses five outer and five inner folds;
CV=10 uses ten outer and ten inner folds. Pool outer-fold predictions for the five headline metrics.
Report the outer mean F1 in `CV_F1_Mean (%)`; inner tuning scores are saved separately. A final grid search
on all cleaned data produces the deployable model but is **not** used to calculate headline metrics.

The same raw input rows and a matching splitter are used for both algorithms within each protocol.
CV-only and holdout metrics evaluate different partitions and are not interchangeable. Comparisons of
several configurations/ratios on an already-inspected test set are exploratory; use training validation
to choose settings and new independent data for confirmatory claims.


## B1. Experiment Configuration and Checkpoint Isolation


In [ ]:
PROTOCOLS = [
    {'name': '70-30 only',    'slug': '7030_only', 'split': 0.30, 'cv': None},
    {'name': '80-20 only',    'slug': '8020_only', 'split': 0.20, 'cv': None},
    {'name': 'CV=5 only',     'slug': 'cv5_only',  'split': None, 'cv': 5},
    {'name': 'CV=10 only',    'slug': 'cv10_only', 'split': None, 'cv': 10},
    {'name': '70-30 + CV=5',  'slug': '7030_cv5',  'split': 0.30, 'cv': 5},
    {'name': '70-30 + CV=10', 'slug': '7030_cv10', 'split': 0.30, 'cv': 10},
    {'name': '80-20 + CV=5',  'slug': '8020_cv5',  'split': 0.20, 'cv': 5},
    {'name': '80-20 + CV=10','slug': '8020_cv10', 'split': 0.20, 'cv': 10},
]
MODELS = ['RandomForest', 'XGBoost']
FIXED_PARAMS = {
    'XGBoost': {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    'RandomForest': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2},
}
# The original grids are preserved. Broader RF parameter tuning is ON HOLD.
PARAM_GRIDS = {
    'XGBoost': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [3, 6, 9],
        'model__learning_rate': [0.01, 0.1, 0.2],
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [6, 10, None],
        'model__min_samples_split': [2, 5],
    },
}

RATIO_TAG = format(float(SMOTE_RATIO), '.12g').replace('.', 'p')
RUN_SPEC = {
    'notebook_version': NOTEBOOK_VERSION, 'data_fingerprint': DATA_FINGERPRINT,
    'smote_ratio': float(SMOTE_RATIO), 'seed': SEED, 'k_features': K_FEATURES,
    'smote_k_neighbors': SMOTE_K_NEIGHBORS, 'protocols': PROTOCOLS,
    'fixed_params': FIXED_PARAMS, 'param_grids': PARAM_GRIDS,
    'splitter': 'StratifiedKFold(shuffle=True, random_state=42)',
    'threshold_policy': 'default predict(); threshold tuning on hold',
    'packages': PACKAGE_VERSIONS, 'python': platform.python_version(),
}
RUN_ID = hashlib.sha256(json.dumps(RUN_SPEC, sort_keys=True).encode()).hexdigest()[:16]
report_dir = f'{BASE_REPORT_DIR}/validated_smote_{RATIO_TAG}/{RUN_ID}'
ckpt_dir = f'{report_dir}/checkpoints_master'
model_dir = f'{BASE_MODEL_DIR}/validated_smote_{RATIO_TAG}/{RUN_ID}'
for directory in [report_dir, ckpt_dir, model_dir]:
    os.makedirs(directory, exist_ok=True)
with open(f'{report_dir}/run_config.json', 'w') as f:
    json.dump(RUN_SPEC, f, indent=2)
pipeline_memory = joblib.Memory(
    location=f'{tempfile.gettempdir()}/shield_pipeline_cache/{RUN_ID}', verbose=0
) if USE_PIPELINE_CACHE else None

# Preserve the preprocessing demonstration without overwriting the original CSVs.
demo_dir = f'{report_dir}/demo_preprocessed_8020'
os.makedirs(demo_dir, exist_ok=True)
X_train_bal_demo.to_csv(f'{demo_dir}/X_train_processed_8020.csv', index=False)
pd.Series(y_train_bal_demo, name=TARGET_COL).to_csv(f'{demo_dir}/y_train_processed_8020.csv', index=False)
X_test_sel_demo.to_csv(f'{demo_dir}/X_test_processed_8020.csv', index=False)
y_test_demo.to_csv(f'{demo_dir}/y_test_processed_8020.csv', index=False)
pd.DataFrame({'Feature': selected_cols_demo}).to_csv(f'{demo_dir}/selected_features.csv', index=False)
df.to_excel(f'{demo_dir}/preprocessed_dataset.xlsx', index=False)

print(f'{len(PROTOCOLS)} protocols x {len(MODELS)} models = {len(PROTOCOLS) * len(MODELS)} total experiments')
print('SMOTE_RATIO:', SMOTE_RATIO)
print('Run ID:', RUN_ID)
print('Reports/checkpoints:', report_dir)


## B2. Experiment Runner (fold-local IG + SMOTE and nested CV-only)


In [ ]:
def make_estimator(model_name, params=None):
    if model_name == 'RandomForest':
        return RandomForestClassifier(**(params or {}), random_state=SEED, n_jobs=1)
    if model_name == 'XGBoost':
        return XGBClassifier(**(params or {}), random_state=SEED, eval_metric='logloss', n_jobs=1)
    raise ValueError(model_name)

def make_training_pipeline(model_name, params=None):
    return ImbPipeline([
        ('ig', make_selector()),
        ('smote', SMOTE(sampling_strategy=SMOTE_RATIO, random_state=SEED,
                        k_neighbors=SMOTE_K_NEIGHBORS)),
        ('model', make_estimator(model_name, params)),
    ], memory=pipeline_memory)

def make_cv(folds):
    return StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)

def make_grid(model_name, folds):
    return GridSearchCV(make_training_pipeline(model_name), PARAM_GRIDS[model_name],
                        cv=make_cv(folds), scoring='f1', n_jobs=N_JOBS,
                        refit=True, error_score='raise')

def check_resampling_feasible(y, context):
    counts = pd.Series(y).value_counts()
    if set(counts.index) != {0, 1}:
        raise ValueError(f'{context}: both target classes must be present.')
    desired_minority = int(SMOTE_RATIO * int(counts[0]))
    if desired_minority <= counts[1]:
        raise ValueError(f'{context}: SMOTE_RATIO={SMOTE_RATIO} would not add minority samples; '
                         'increase the ratio for this dataset/fold.')
    if counts[1] <= SMOTE_K_NEIGHBORS:
        raise ValueError(f'{context}: too few bankrupt samples for SMOTE k_neighbors={SMOTE_K_NEIGHBORS}.')

def check_cv_feasible(X, y, folds, context):
    counts = pd.Series(y).value_counts()
    if counts.min() < folds:
        raise ValueError(f'{context}: fewer minority samples than CV folds.')
    for fold_id, (train_idx, _) in enumerate(make_cv(folds).split(X, y), start=1):
        check_resampling_feasible(y.iloc[train_idx], f'{context}, training fold {fold_id}')

def compute_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy (%)': accuracy_score(y_true, y_pred) * 100,
        'Precision (%)': precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'Recall (%)': recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'F1-Score (%)': f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'ROC-AUC (%)': roc_auc_score(y_true, y_prob) * 100,
    }

def predict_outputs(fitted_pipeline, X):
    classes = fitted_pipeline.named_steps['model'].classes_.tolist()
    positive_idx = classes.index(1)
    return fitted_pipeline.predict(X), fitted_pipeline.predict_proba(X)[:, positive_idx]

def selected_features(fitted_pipeline):
    selector = fitted_pipeline.named_steps['ig']
    return selector.feature_names_in_[selector.get_support()].tolist()

def checkpoint_paths(slug, model_name):
    base = f'{ckpt_dir}/{slug}_{model_name}'
    return {'result': f'{base}_result.json', 'preds': f'{base}_preds.npz',
            'model': f'{base}_model.joblib'}

def load_checkpoint(slug, model_name):
    paths = checkpoint_paths(slug, model_name)
    if not all(os.path.exists(p) for p in paths.values()):
        return None
    with open(paths['result']) as f:
        result = json.load(f)
    if result.get('Run_ID') != RUN_ID:
        return None
    with np.load(paths['preds'], allow_pickle=False) as arrays:
        return result, arrays['y_true'].copy(), arrays['y_pred'].copy(), arrays['y_prob'].copy()

def save_checkpoint(slug, model_name, result, y_true, y_pred, y_prob, fitted_model):
    paths = checkpoint_paths(slug, model_name)
    # Save the result last: it marks an experiment as complete.
    joblib.dump(fitted_model, paths['model'])
    np.savez_compressed(paths['preds'], y_true=np.asarray(y_true),
                        y_pred=np.asarray(y_pred), y_prob=np.asarray(y_prob))
    with open(paths['result'], 'w') as f:
        json.dump(result, f, indent=2)

def run_nested_cv(model_name, config):
    folds = config['cv']
    y_pred = np.empty(len(y_full), dtype=int)
    y_prob = np.empty(len(y_full), dtype=float)
    visited = np.zeros(len(y_full), dtype=bool)
    fold_records = []
    for fold_id, (train_idx, val_idx) in enumerate(make_cv(folds).split(X_full, y_full), start=1):
        fold_base = f"{ckpt_dir}/{config['slug']}_{model_name}_outer_{fold_id}"
        fold_json, fold_npz = f'{fold_base}.json', f'{fold_base}.npz'
        cached = None
        if os.path.exists(fold_json) and os.path.exists(fold_npz):
            with open(fold_json) as f:
                record = json.load(f)
            if record.get('Run_ID') == RUN_ID:
                with np.load(fold_npz, allow_pickle=False) as arrays:
                    if np.array_equal(arrays['val_idx'], val_idx):
                        cached = arrays['y_pred'].copy(), arrays['y_prob'].copy(), record
        if cached is not None:
            fold_pred, fold_prob, record = cached
            print(f'  [OUTER CHECKPOINT] {model_name} | {config["name"]} | fold {fold_id}/{folds}')
        else:
            print(f'  [NESTED CV] {model_name} | {config["name"]} | outer fold {fold_id}/{folds}')
            X_outer_train, y_outer_train = X_full.iloc[train_idx], y_full.iloc[train_idx]
            X_outer_val, y_outer_val = X_full.iloc[val_idx], y_full.iloc[val_idx]
            check_cv_feasible(X_outer_train, y_outer_train, folds, 'Nested inner CV')
            inner_grid = make_grid(model_name, folds)
            inner_grid.fit(X_outer_train, y_outer_train)
            fold_pred, fold_prob = predict_outputs(inner_grid.best_estimator_, X_outer_val)
            record = {'Run_ID': RUN_ID, 'Fold': fold_id,
                      'Best_Params': inner_grid.best_params_,
                      'Inner_CV_F1 (%)': inner_grid.best_score_ * 100,
                      'Selected_Features': selected_features(inner_grid.best_estimator_),
                      'Train_Count': len(train_idx), 'Validation_Count': len(val_idx),
                      **compute_metrics(y_outer_val, fold_pred, fold_prob)}
            np.savez_compressed(fold_npz, val_idx=val_idx, y_pred=fold_pred, y_prob=fold_prob)
            with open(fold_json, 'w') as f:
                json.dump(record, f, indent=2)
        assert not visited[val_idx].any(), 'An outer validation row was predicted twice.'
        y_pred[val_idx], y_prob[val_idx], visited[val_idx] = fold_pred, fold_prob, True
        fold_records.append(record)
    assert visited.all(), 'Every row must receive exactly one outer held-out prediction.'
    pd.DataFrame(fold_records).to_csv(
        f"{report_dir}/{config['slug']}_{model_name}_outer_fold_results.csv", index=False)

    # Final refit for deployment ONLY. These tuning scores never replace outer OOF metrics.
    print(f'  [FINAL REFIT] {model_name} | {config["name"]} | all cleaned data')
    check_cv_feasible(X_full, y_full, folds, 'Final full-data CV')
    final_grid = make_grid(model_name, folds)
    final_grid.fit(X_full, y_full)
    return y_pred, y_prob, fold_records, final_grid

def run_experiment(model_name, config):
    split_size, cv_folds = config['split'], config['cv']
    result = {'Model': model_name, 'Experiment': config['name'], 'Split': split_size, 'CV': cv_folds,
              'SMOTE_Ratio': float(SMOTE_RATIO), 'Run_ID': RUN_ID}

    if split_size is None:
        y_pred, y_prob, fold_records, final_grid = run_nested_cv(model_name, config)
        y_true = y_full.to_numpy()
        fitted_model = final_grid.best_estimator_
        outer_scores = [r['F1-Score (%)'] for r in fold_records]
        result.update(compute_metrics(y_true, y_pred, y_prob))
        result.update({'Best_Params': final_grid.best_params_,
                       'CV_F1_Mean (%)': float(np.mean(outer_scores)),
                       'CV_F1_Std (%)': float(np.std(outer_scores, ddof=1)),
                       'Inner_CV_F1_Mean (%)': float(np.mean([r['Inner_CV_F1 (%)'] for r in fold_records])),
                       'Final_Refit_CV_F1 (%)': final_grid.best_score_ * 100,
                       'Evaluation': 'Nested CV outer out-of-fold predictions'})
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X_full, y_full, test_size=split_size, random_state=SEED, stratify=y_full)
        if cv_folds is None:
            check_resampling_feasible(y_train, 'Holdout training set')
            fitted_model = make_training_pipeline(model_name, FIXED_PARAMS[model_name])
            fitted_model.fit(X_train, y_train)
            result['Best_Params'] = FIXED_PARAMS[model_name]
            result['CV_F1_Mean (%)'] = None
        else:
            check_cv_feasible(X_train, y_train, cv_folds, 'Holdout training CV')
            grid = make_grid(model_name, cv_folds)
            grid.fit(X_train, y_train)
            fitted_model = grid.best_estimator_
            result['Best_Params'] = grid.best_params_
            result['CV_F1_Mean (%)'] = grid.best_score_ * 100
        y_pred, y_prob = predict_outputs(fitted_model, X_test)
        y_true = y_test.to_numpy()
        result.update(compute_metrics(y_true, y_pred, y_prob))
        result['Evaluation'] = 'Untouched holdout test set'

    result['Selected_Features'] = selected_features(fitted_model)
    result['Evaluation_Count'] = len(y_true)
    result['Bankrupt_Support'] = int(np.sum(y_true == 1))
    return result, y_true, y_pred, y_prob, fitted_model

print('Runner defined: IG → SMOTE → model inside every CV training fold.')
print('CV-only uses outer evaluation + inner tuning; threshold tuning is ON HOLD.')


## B3. Run All 16 Experiments
Start with **RandomForest | 80-20 + CV=10**, then complete the other 15. Existing checkpoints from the
exact same configuration are reused. The final table is ordered like the original notebook.
First-run nested CV is substantially more expensive than ordinary CV.


In [ ]:
all_results = []
predictions = {cfg['slug']: {} for cfg in PROTOCOLS}
baseline_order = [(cfg, model_name) for cfg in PROTOCOLS for model_name in MODELS]
execution_order = sorted(baseline_order, key=lambda item: 0 if (
    item[0]['slug'] == '8020_cv10' and item[1] == 'RandomForest') else 1)

for cfg, model_name in execution_order:
    ckpt = load_checkpoint(cfg['slug'], model_name)
    if ckpt is not None:
        print(f"[CHECKPOINT] Skipping {model_name} | {cfg['name']} (already done)")
        result, y_true, y_pred, y_prob = ckpt
    else:
        print(f"[RUNNING] {model_name} | {cfg['name']} ...")
        result, y_true, y_pred, y_prob, fitted_model = run_experiment(model_name, cfg)
        save_checkpoint(cfg['slug'], model_name, result, y_true, y_pred, y_prob, fitted_model)
    print(f"  Accuracy: {result['Accuracy (%)']:.2f}% | Precision: {result['Precision (%)']:.2f}% | "
          f"Recall: {result['Recall (%)']:.2f}% | F1: {result['F1-Score (%)']:.2f}% | "
          f"ROC-AUC: {result['ROC-AUC (%)']:.2f}%")
    all_results.append(result)
    predictions[cfg['slug']][model_name] = (y_true, y_pred, y_prob)

protocol_order = {cfg['name']: i for i, cfg in enumerate(PROTOCOLS)}
model_order = {name: i for i, name in enumerate(MODELS)}
results_df = pd.DataFrame(all_results)
results_df['_protocol_order'] = results_df['Experiment'].map(protocol_order)
results_df['_model_order'] = results_df['Model'].map(model_order)
results_df = results_df.sort_values(['_protocol_order', '_model_order']).drop(
    columns=['_protocol_order', '_model_order']).reset_index(drop=True)
# Keep the original headline column order; new traceability columns follow it.
original_columns = ['Model', 'Experiment', 'Split', 'CV', 'Accuracy (%)', 'Precision (%)',
                    'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)', 'Best_Params', 'CV_F1_Mean (%)']
results_df = results_df[original_columns + [c for c in results_df if c not in original_columns]]
print('\n=== ALL 16 EXPERIMENTS COMPLETE ===')
display(results_df[original_columns])


## B4. Full Metrics Table and Classification Reports
*The same report layout and target labels as the original.*


In [ ]:
for cfg in PROTOCOLS:
    for model_name in MODELS:
        y_true, y_pred, y_prob = predictions[cfg['slug']][model_name]
        print(f"\n=== Classification Report: {model_name} | {cfg['name']} ===")
        print(classification_report(y_true, y_pred, labels=[0, 1],
              target_names=['Not Bankrupt', 'Bankrupt'], zero_division=0))


## B5. Charts — Bar Comparison, Confusion Matrices, ROC Curves
The original 24-chart layout and filenames are preserved inside this run's isolated output directory.
`Healthy (0)` in the confusion matrices is the original display label for **Not Bankrupt**;
it is not an independently verified financial-health diagnosis. Titles distinguish nested CV evaluation.


In [ ]:
MODEL_COLORS = {'RandomForest': '#4C93E0', 'XGBoost': '#D866F5'}
for cfg in PROTOCOLS:
    slug, pname = cfg['slug'], cfg['name']
    eval_suffix = ' [Nested outer OOF]' if cfg['split'] is None else ''
    row_rf = results_df[(results_df['Experiment'] == pname) & (results_df['Model'] == 'RandomForest')].iloc[0]
    row_xgb = results_df[(results_df['Experiment'] == pname) & (results_df['Model'] == 'XGBoost')].iloc[0]
    y_true_rf, y_pred_rf, y_prob_rf = predictions[slug]['RandomForest']
    y_true_xgb, y_pred_xgb, y_prob_xgb = predictions[slug]['XGBoost']

    metrics_labels = ['Accuracy (%)', 'ROC-AUC (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)']
    rf_vals, xgb_vals = [row_rf[m] for m in metrics_labels], [row_xgb[m] for m in metrics_labels]
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(metrics_labels)); width = 0.35
    b1 = ax.bar(x - width/2, rf_vals, width, label='Random Forest', color=MODEL_COLORS['RandomForest'])
    b2 = ax.bar(x + width/2, xgb_vals, width, label='XGBoost', color=MODEL_COLORS['XGBoost'])
    for bars in (b1, b2):
        for b in bars:
            ax.annotate(f'{b.get_height():.1f}%', (b.get_x()+b.get_width()/2, b.get_height()),
                        textcoords='offset points', xytext=(0,3), ha='center', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(metrics_labels); ax.set_ylabel('Score (%)'); ax.set_ylim(0,110)
    ax.set_title(f'Model Performance Comparison — {pname}{eval_suffix} (Target: Bankrupt SMEs)', weight='bold')
    ax.legend(); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_bar_comparison.png', dpi=300); plt.show(); plt.close()

    cm_rf, cm_xgb = confusion_matrix(y_true_rf, y_pred_rf, labels=[0,1]), confusion_matrix(y_true_xgb, y_pred_xgb, labels=[0,1])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, cm, title, cmap in zip(axes, [cm_rf, cm_xgb],
                                 ['Random Forest Confusion Matrix', 'XGBoost Confusion Matrix'],
                                 ['Blues', 'Purples']):
        ax.imshow(cm, cmap=cmap)
        ax.set_title(title, weight='bold')
        ax.set_xticks([0,1]); ax.set_xticklabels(['Healthy (0)', 'Bankrupt (1)'])
        ax.set_yticks([0,1]); ax.set_yticklabels(['Healthy (0)', 'Bankrupt (1)'])
        ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                        color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
    plt.suptitle(f'Confusion Matrices — {pname}{eval_suffix}', weight='bold'); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_confusion_matrices.png', dpi=300); plt.show(); plt.close()

    fpr_rf, tpr_rf, _ = roc_curve(y_true_rf, y_prob_rf)
    fpr_xgb, tpr_xgb, _ = roc_curve(y_true_xgb, y_prob_xgb)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr_rf, tpr_rf, color=MODEL_COLORS['RandomForest'], label=f"Random Forest (AUC = {row_rf['ROC-AUC (%)']/100:.4f})")
    ax.plot(fpr_xgb, tpr_xgb, color=MODEL_COLORS['XGBoost'], label=f"XGBoost (AUC = {row_xgb['ROC-AUC (%)']/100:.4f})")
    ax.plot([0,1], [0,1], linestyle='--', color='gray')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Receiver Operating Characteristic (ROC) Curve — {pname}{eval_suffix}', weight='bold')
    ax.legend(loc='lower right'); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_roc_curve.png', dpi=300); plt.show(); plt.close()
    print(f'Saved 3 charts for: {pname}')


## B6. Save Final Result Tables
The headline slide table keeps the original column layout. Full results retain traceability and fold details.


In [ ]:
results_df.to_csv(f'{report_dir}/master_results_full.csv', index=False)
slide_columns = ['Model', 'Experiment', 'Split', 'CV', 'Accuracy (%)', 'Precision (%)',
                 'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)', 'CV_F1_Mean (%)']
slide_df = results_df[slide_columns]
slide_df.to_csv(f'{report_dir}/master_results_slide.csv', index=False)
with open(f'{report_dir}/master_results_full.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('=== FINAL RESULTS TABLE ===')
display(slide_df)
print(f'\nAll tables, charts (24 images), and model checkpoints (16 .joblib files) saved to:')
print(f'  {report_dir}')
print('\nReview CV and held-out results before selecting a final model. The previous winner is a candidate, not an assumed new winner.')


## B7. Export the RF 80:20 + CV=10 Candidate and Its Fixed Input Schema
This exports the requested starting candidate; it does **not** automatically claim it is the best model
for the new ratio. Change the two selection variables only after reviewing results.

Save: (1) the complete IG → SMOTE → model pipeline, which expects the original cleaned predictor columns;
(2) the final classifier accepting exactly the selected 20 columns; (3) a feature manifest with their exact
names and order; (4) the fitted IG selector. The frontend uses the fixed final list and does not rerun IG.
Files are written in a new isolated directory and do not replace the prototype's deployed model.


In [ ]:
EXPORT_PROTOCOL_SLUG = '8020_cv10'
EXPORT_MODEL_NAME = 'RandomForest'
source_paths = checkpoint_paths(EXPORT_PROTOCOL_SLUG, EXPORT_MODEL_NAME)
chosen_checkpoint = load_checkpoint(EXPORT_PROTOCOL_SLUG, EXPORT_MODEL_NAME)
if chosen_checkpoint is None:
    raise RuntimeError('Run the selected experiment before exporting it.')
chosen_result = chosen_checkpoint[0]
chosen_pipeline = joblib.load(source_paths['model'])
final_selector = chosen_pipeline.named_steps['ig']
final_classifier = chosen_pipeline.named_steps['model']
final_features = selected_features(chosen_pipeline)
assert len(final_features) == K_FEATURES
assert final_classifier.feature_names_in_.tolist() == final_features, 'Classifier input order differs from selector output.'

# Verify the 20-input deployment classifier matches the complete training pipeline.
verification_input = X_full.iloc[:min(32, len(X_full))]
np.testing.assert_allclose(
    chosen_pipeline.predict_proba(verification_input),
    final_classifier.predict_proba(final_selector.transform(verification_input)),
    rtol=1e-10, atol=1e-12)

stem = f'SHIELD_{EXPORT_MODEL_NAME}_{EXPORT_PROTOCOL_SLUG}_SMOTE_{RATIO_TAG}'
full_pipeline_path = f'{model_dir}/{stem}_full_pipeline.joblib'
classifier_path = f'{model_dir}/{stem}_20_feature_classifier.joblib'
selector_path = f'{model_dir}/{stem}_ig_selector.joblib'
joblib.dump(chosen_pipeline, full_pipeline_path)
joblib.dump(final_classifier, classifier_path)
joblib.dump(final_selector, selector_path)
manifest = {
    'Run_ID': RUN_ID, 'Model': EXPORT_MODEL_NAME, 'Protocol': EXPORT_PROTOCOL_SLUG,
    'SMOTE_Ratio': float(SMOTE_RATIO), 'Target': TARGET_COL, 'Label_Map': LABEL_MAP,
    'Decision_Rule': 'default classifier.predict(); no threshold tuning',
    'Full_Pipeline_Input_Columns': final_selector.feature_names_in_.tolist(),
    'Deployment_Classifier_Input_Columns': final_features,
    'Selected_Feature_IG_Scores': {
        str(name): float(score) for name, score, keep in zip(
            final_selector.feature_names_in_, final_selector.scores_, final_selector.get_support()) if keep
    },
    'Experiment_Result': chosen_result,
}
with open(f'{model_dir}/{stem}_feature_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
pd.DataFrame({'Input_Order': range(1, K_FEATURES+1), 'Feature': final_features}).to_csv(
    f'{model_dir}/{stem}_selected_features.csv', index=False)
print(f'SUCCESS: candidate exported ({EXPORT_MODEL_NAME} | {EXPORT_PROTOCOL_SLUG}, SMOTE={SMOTE_RATIO}).')
print('Full pipeline:', full_pipeline_path)
print('20-input classifier:', classifier_path)
print('Fixed frontend/backend input schema:')
for index, feature in enumerate(final_features, start=1):
    print(f'  {index:2d}. {feature}')
print('This candidate has not replaced the deployed prototype model.')


## Method Notes and References
- Split-only runs use the original fixed parameters; CV runs tune the original grids using F1.
  Differences therefore include both partitioning and parameter tuning, not partitioning alone.
- CV-only headline scores use pooled outer held-out predictions. Their mean fold F1 can differ from pooled F1.
  Parameter choices may differ across outer folds; per-fold parameters/features are saved in separate CSVs.
- Every transformation that learns from targets (IG) and all synthetic sampling fit only the applicable training data.
- Seeded splits and package/config/data fingerprints support reproducibility. Change `NOTEBOOK_VERSION` if you
  manually change algorithm logic so old checkpoints cannot be reused accidentally.
- If predictors have materially different scales, inspect SMOTE's distance assumptions before adding scaling.
  Scaling and new cleaning rules are not silently introduced here.
- The current thresholds used for prototype severity bands are separate from this notebook's default binary prediction.
- [Scikit-learn: preprocessing leakage and pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- [Scikit-learn: nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [Imbalanced-learn: SMOTE sampling ratios](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html)
